# 2 · Run CBM-CFS3

Imports each stack written by notebook 1 into a CBM-CFS3 project with the **Standard Import Tool**, then runs **Makelist** and the **CBM-CFS3 simulation** with [`cbm3_python`](https://github.com/cat-cfs/cbm3_python).

**Requirements**
- Windows with the [Operational-Scale CBM-CFS3](https://natural-resources.canada.ca/climate-change/climate-change-impacts-forests/carbon-budget-model) toolbox installed
- Microsoft Access Database Engine with the same bitness as Python
- `pip install git+https://github.com/cat-cfs/cbm3_python.git`

Each stack takes tens of minutes. The results database for a stack is 500–700 MB.

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
import pyodbc
from cbm3_python import toolbox_defaults
from cbm3_python.cbm3data import sit_helper
from cbm3_python.simulation import projectsimulator

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUNS = Path(os.environ.get('CBM_RUNS_DIR', ROOT / 'runs'))   # one folder per forest, one sub-folder per stack

FORESTS = os.environ.get('CBM_FORESTS', 'wabigoon,troutlake').split(',')
N_TIMESTEPS = int(os.environ.get('CBM_TIMESTEPS', 30))   # 2000 + 30 years; the story page reads 2000-2010
OVERWRITE = False

print('CBM-CFS3 toolbox:', toolbox_defaults.get_install_path())

## Import and simulate every stack

`csv_import` reads `sit_*.csv` and `mapping.json` from **the stack's own folder** and writes `project.mdb` next to them; `projectsimulator.run` writes `results.mdb` and the CBM log to the same folder.

In [ ]:
def run_stack(folder):
    project, results = folder / 'project.mdb', folder / 'results.mdb'
    if results.exists() and not OVERWRITE:
        print(f'  {folder.relative_to(RUNS)}: results.mdb exists, skipped')
        return
    missing = [f for f in ['sit_inventory.csv', 'sit_classifiers.csv', 'sit_yield.csv', 'sit_events.csv', 'sit_transitions.csv',
                           'sit_age_classes.csv', 'sit_disturbance_types.csv', 'mapping.json'] if not (folder / f).exists()]
    if missing:
        raise FileNotFoundError(f'{folder}: missing {missing}; run notebook 1 first')
    for old in (project, results):
        old.unlink(missing_ok=True)

    t0 = time.time()
    sit_helper.csv_import(csv_dir=str(folder), imported_project_path=str(project), working_dir=str(folder))
    print(f'  {folder.relative_to(RUNS)}: imported in {time.time() - t0:.0f} s')
    projectsimulator.run(project_path=str(project), results_database_path=str(results), n_timesteps=N_TIMESTEPS,
                         tempfiles_output_dir=str(folder / 'temp'), skip_makelist=False,
                         stdout_path=str(folder / 'cbm_log.txt'))
    print(f'  {folder.relative_to(RUNS)}: simulated {N_TIMESTEPS} steps in {(time.time() - t0) / 60:.1f} min')


for key in FORESTS:
    stacks = sorted((RUNS / key).glob('stack*'))
    print(f'{key}: {len(stacks)} stacks')
    for folder in stacks:
        run_stack(folder)

## Check the runs before using them

Each results database should hold its own stack: the simulated area must match that stack's inventory, and no stand may appear in two stacks. A stack imported from the wrong folder shows up here as an area mismatch or a full overlap.

In [ ]:
def query(path, sql):
    with pyodbc.connect(r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=' + str(path)) as conn:
        return conn.cursor().execute(sql).fetchall()


rows = []
for key in FORESTS:
    seen = {}
    for folder in sorted((RUNS / key).glob('stack*')):
        results = folder / 'results.mdb'
        if not results.exists():
            continue
        area = query(results, 'SELECT SUM(Area) FROM tblAgeIndicators WHERE TimeStep = 0')[0][0]
        ids = {r[0] for r in query(results, 'SELECT UserDefdSubClassName FROM tblUserDefdSubclasses WHERE UserDefdClassID = 2')}
        expected = pd.read_csv(folder / 'sit_inventory.csv').Area.sum()
        overlap = max((len(ids & other) for other in seen.values()), default=0)
        seen[folder.name] = ids
        rows.append(dict(forest=key, stack=folder.name, simulated_ha=round(area), inventory_ha=round(expected),
                         area_matches=abs(area - expected) < 1, stands=len(ids), shared_with_earlier_stacks=overlap))
checks = pd.DataFrame(rows)
assert checks.area_matches.all() and (checks.shared_with_earlier_stacks == 0).all(), 'a stack does not hold its own stands'
checks